# Packages

In [ ]:
%pip install deepeval accelerate

# Imports

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM
import asyncio
import csv

# Load Test Cases

In [ ]:
csv_data = [
    ["input", "actual_output", "retrieval_context"],
    ["What is the capital of France?", "Paris", "Paris is the capital of France."],
    ["Who painted the Mona Lisa?", "Leonardo da Vinci painted the Mona Lisa, a famous portrait in the Louvre Museum.", "Leonardo da Vinci was an Italian polymath."],
    ["What is the tallest mountain in the world?", "Mount Everest is the Earth's highest mountain above sea level.", "Mount Everest is located in the Himalayas and is the highest mountain above sea level."]
]

with open("content.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

test_cases = []
with open("content.csv", "r") as f:
    reader = csv.reader(f)

    next(reader, None)

    for values in reader:
        if len(values) == 3:
            values = [val.strip() for val in values]

            test_cases.append(
                LLMTestCase(
                    input=values[0],
                    actual_output=values[1],
                    retrieval_context=[values[2]]
                )
            )

print(f"Loaded {len(test_cases)} test cases from file")

# Evaluation Model

In [ ]:
class EvalModel(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.pipe = None
        self.load_model(model, tokenizer)

    def load_model(self, model, tokenizer):
        self.pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
        )

        return self.pipe

    def _generate(self, prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]

        outputs = self.pipe(
            messages,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=self.pipe.tokenizer.eos_token_id
        )

        return outputs[0]["generated_text"][-1]["content"].strip()

    # Strip kwargs at method signature level
    def generate(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        return self._generate(prompt)

    async def a_generate(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        loop = asyncio.get_running_loop()
        return await loop.run_in_executor(None, self._generate, prompt)

    def generate_raw_response(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        response = self._generate(prompt)

        return (response, 0.0)

    async def a_generate_raw_response(self, prompt: str, top_logprobs=None, logprobs=None, **kwargs) -> str:
        loop = asyncio.get_running_loop()

        response = await loop.run_in_executor(None, self._generate, prompt)

        return (response, 0.0)

    def get_model_name(self) -> str:
        return "Eval Model"

# Evaluate

In [ ]:
print("Evaluating")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

evaluator = EvalModel(model, tokenizer)

results = evaluate(
    test_cases,
    [
      GEval(
          name="Factuality & Grounding",
          criteria="Determine if actual_output is factually correct and fully grounded in retrieval_context.",
          evaluation_steps=[
              "List all factual claims made in the actual_output.",
              "Verify each claim is directly supported by retrieval_context.",
              "Penalize hallucinations, speculation, or unsupported details.",
              "Score 1-10: 10=perfectly factual/grounded, 1=fully hallucinated."
          ],
          evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
          model=evaluator,
          threshold=0.7
      ),
      GEval(
          name="Recall",
          criteria="Assess if retrieval_context contains all information needed to fully answer the input query.",
          evaluation_steps=[
              "Identify key facts required to answer the input query.",
              "Check if retrieval_context includes these key facts.",
              "Penalize missing critical information needed for complete answer.",
              "Score 1-10: 10=context fully supports ideal answer, 1=critical info missing."
          ],
          evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT],
          model=evaluator,
          threshold=0.7
      )
    ]
)

In [ ]:
print("\n=== RESULTS ===")
print(f"{'Case':<4} {'Fact':<6} {'Rec':<6} {'Pass':<6} {'Reason':<50}")
print("-" * 90)

for idx, result in enumerate(results.test_results):
  fact_score = result.metrics_data[0].score  # Factuality
  recall_score = result.metrics_data[1].score  # Recall
  fact_reason = result.metrics_data[0].reason[:45]

  status = "Y" if fact_score >= 0.7 and recall_score >= 0.7 else "N"

  print(f"{idx:<4} {fact_score:<6.3f} {recall_score:<6.3f} {status:<6} {fact_reason}")

fact_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7) / len(results.test_results)
recall_pass = sum(1 for r in results.test_results if r.metrics_data[1].score >= 0.7) / len(results.test_results)
both_pass = sum(1 for r in results.test_results if r.metrics_data[0].score >= 0.7 and r.metrics_data[1].score >= 0.7) / len(results.test_results)

print(f"\nSummary:")
print(f"Factuality Pass Rate: {fact_pass:.1%}")
print(f"Recall Pass Rate: {recall_pass:.1%}")
print(f"Both Pass Rate: {both_pass:.1%}")